In [1]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, roc_auc_score, roc_curve, confusion_matrix)
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Конфигурация
models_paths = {
    'InceptionV3': '../train_test_v2_balanced/InceptionV3_Deepfake.keras',
    'MobileNetV2': '../train_test_v2_balanced/MobileNetV2_Deepfake.keras',
    'ResNetV2': '../train_test_v2_balanced/ResNetV2_Deepfake.keras',
    'VGG16': '../train_test_v2_balanced/VGG16_Deepfake.keras',
    'Xception': '../train_test_v2_balanced/Xception_Deepfake.keras'
}

preprocess_functions = [
    tf.keras.applications.inception_v3.preprocess_input,
    tf.keras.applications.mobilenet_v2.preprocess_input,
    tf.keras.applications.resnet_v2.preprocess_input,
    tf.keras.applications.vgg16.preprocess_input,
    tf.keras.applications.xception.preprocess_input
]

# Функция для усреднения предсказаний по последовательности
def average_predictions_by_sequence(predictions, sequence_length=10):
    n_samples = len(predictions)
    n_sequences = n_samples // sequence_length
    
    # Убедимся, что количество сэмплов кратно длине последовательности
    if n_samples % sequence_length != 0:
        print(f"Внимание: {n_samples} сэмплов не делится нацело на {sequence_length}")
        print(f"Будет использовано {n_sequences} полных последовательностей")
    
    # Усредняем предсказания по последовательностям
    averaged = []
    sequence_indices = []
    
    for i in range(n_sequences):
        start_idx = i * sequence_length
        end_idx = start_idx + sequence_length
        sequence_preds = predictions[start_idx:end_idx]
        
        # Усредняем по кадрам в последовательности
        avg_pred = np.mean(sequence_preds, axis=0)
        averaged.append(avg_pred)
        sequence_indices.append(start_idx)
    
    return np.array(averaged), np.array(sequence_indices)

# Функция для вычисления оптимального порога по методу Йудена
def find_optimal_threshold(y_true, y_pred_proba):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
    youden_index = tpr - fpr
    optimal_idx = np.argmax(youden_index)
    optimal_threshold = thresholds[optimal_idx]
    
    return optimal_threshold, youden_index[optimal_idx], fpr, tpr

# Функция для вычисления метрик
def calculate_metrics(y_true, y_pred, y_pred_proba=None):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='binary'),
        'recall': recall_score(y_true, y_pred, average='binary'),
        'f1': f1_score(y_true, y_pred, average='binary'),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist()
    }
    
    # Если есть вероятности, вычисляем ROC-AUC
    if y_pred_proba is not None:
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba)
        except:
            metrics['roc_auc'] = np.nan
    
    # Дополнительные метрики из confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metrics['true_negative'] = int(tn)
    metrics['false_positive'] = int(fp)
    metrics['false_negative'] = int(fn)
    metrics['true_positive'] = int(tp)
    
    # Специфичность (Specificity)
    metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return metrics

In [2]:
# Основной цикл обработки
results = []
all_predictions = {}
all_optimal_thresholds = {}

# Сначала получим true_labels для всех сэмплов
print("Загрузка тестовых данных...")
i = 0
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_functions[0]  # используем первую для загрузки
)

test_generator = test_datagen.flow_from_directory(
    '../DeepfakeTIMIT/test_data',
    batch_size=32,
    shuffle=False,  # важно отключить перемешивание для сохранения порядка
    target_size=(224, 224)
)

true_labels = test_generator.classes
print(f"Загружено {len(true_labels)} тестовых изображений")

# Предполагаем длину последовательности (уточните это значение!)
SEQUENCE_LENGTH = 10  # Измените на актуальное значение

print(f"\nПредполагаемая длина последовательности: {SEQUENCE_LENGTH} кадров")
print(f"Ожидаемое количество последовательностей: {len(true_labels) // SEQUENCE_LENGTH}")

# Обрабатываем каждую модель
for idx, (name, path) in enumerate(models_paths.items()):
    print(f"\n{'='*60}")
    print(f"Обработка модели: {name}")
    print(f"Путь: {path}")
    print('='*60)
    
    # Создаем ImageDataGenerator с соответствующей предобработкой
    test_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_functions[idx]
    )
    
    test_generator = test_datagen.flow_from_directory(
        '../DeepfakeTIMIT/test_data',
        batch_size=32,
        shuffle=False,  # важно сохранить порядок
        target_size=(224, 224),
        class_mode='binary' if 'binary' in dir(test_generator) else 'categorical'
    )
    
    # Загружаем модель
    print("Загрузка модели...")
    model = tf.keras.models.load_model(path)
    
    # Получаем предсказания
    print("Получение предсказаний...")
    predictions = model.predict(test_generator, verbose=1)
    
    # Проверяем форму предсказаний
    print(f"Форма предсказаний: {predictions.shape}")
    
    # Если модель бинарная и выдает один выход (n_samples, 1), преобразуем в (n_samples, 2)
    if len(predictions.shape) == 1:
        predictions = predictions.reshape(-1, 1)
        # Для бинарной классификации создаем второй класс
        predictions = np.concatenate([1 - predictions, predictions], axis=1)
    elif predictions.shape[1] == 1:
        # Аналогично для (n_samples, 1)
        predictions = np.concatenate([1 - predictions, predictions], axis=1)
    
    # Усредняем предсказания по последовательностям
    print(f"Усреднение предсказаний по последовательностям (длина={SEQUENCE_LENGTH})...")
    predictions_avg, sequence_indices = average_predictions_by_sequence(
        predictions, 
        sequence_length=SEQUENCE_LENGTH
    )
    
    # Получаем true_labels для последовательностей
    # Берем метку первого кадра в каждой последовательности
    true_labels_sequences = true_labels[sequence_indices]
    
    # Берем вероятность класса 1 (deepfake)
    if predictions_avg.shape[1] > 1:
        y_pred_proba = predictions_avg[:, 1]
    else:
        y_pred_proba = predictions_avg.flatten()
    
    # Находим оптимальный порог
    print("Поиск оптимального порога...")
    optimal_threshold, youden_index, fpr, tpr = find_optimal_threshold(
        true_labels_sequences, 
        y_pred_proba
    )
    
    print(f"Оптимальный порог: {optimal_threshold:.4f}")
    print(f"Индекс Йудена: {youden_index:.4f}")
    
    # Применяем порог для получения бинарных предсказаний
    y_pred_binary = (y_pred_proba >= optimal_threshold).astype(int)
    
    # Вычисляем метрики
    print("Вычисление метрик...")
    metrics = calculate_metrics(true_labels_sequences, y_pred_binary, y_pred_proba)
    
    # Сохраняем результаты
    model_results = {
        'model_name': name,
        'optimal_threshold': optimal_threshold,
        'youden_index': youden_index,
        'n_sequences': len(true_labels_sequences),
        'predictions_shape': predictions.shape,
        'predictions_avg_shape': predictions_avg.shape,
        **metrics
    }
    
    results.append(model_results)
    
    # Сохраняем предсказания и пороги для будущего использования
    all_predictions[name] = {
        'per_frame': predictions,
        'per_sequence': predictions_avg,
        'binary': y_pred_binary,
        'probabilities': y_pred_proba
    }
    
    all_optimal_thresholds[name] = optimal_threshold
    
    # Выводим краткую статистику
    print(f"\nРезультаты для {name}:")
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1-Score: {metrics['f1']:.4f}")
    if 'roc_auc' in metrics:
        print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"  Confusion Matrix: TP={metrics['true_positive']}, FP={metrics['false_positive']}, "
          f"FN={metrics['false_negative']}, TN={metrics['true_negative']}")

Загрузка тестовых данных...
Found 20000 images belonging to 2 classes.
Загружено 20000 тестовых изображений

Предполагаемая длина последовательности: 10 кадров
Ожидаемое количество последовательностей: 2000

Обработка модели: InceptionV3
Путь: ../train_test_v2_balanced/InceptionV3_Deepfake.keras
Found 20000 images belonging to 2 classes.
Загрузка модели...
Получение предсказаний...
625/625 [==============================] - 252s 396ms/step
Форма предсказаний: (20000, 1)
Усреднение предсказаний по последовательностям (длина=10)...
Поиск оптимального порога...
Оптимальный порог: 0.8365
Индекс Йудена: 0.2450
Вычисление метрик...

Результаты для InceptionV3:
  Accuracy: 0.6225
  Precision: 0.5801
  Recall: 0.8870
  F1-Score: 0.7015
  ROC-AUC: 0.5782
  Confusion Matrix: TP=887, FP=642, FN=113, TN=358

Обработка модели: MobileNetV2
Путь: ../train_test_v2_balanced/MobileNetV2_Deepfake.keras
Found 20000 images belonging to 2 classes.
Загрузка модели...
Получение предсказаний...
625/625 [======

In [ ]:
# Сохраняем все результаты в DataFrame
print(f"\n{'='*60}")
print("Сводка результатов всех моделей")
print('='*60)

df_results = pd.DataFrame(results)
print(df_results[['model_name', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'optimal_threshold']])

# Сохраняем результаты в файлы
print("\nСохранение результатов...")

# 1. Сохраняем таблицу с метриками
df_results.to_csv('balanced_DeepfakeTIMIT_data_results_v2/model_metrics_per_sequence.csv', index=False)
df_results.to_excel('balanced_DeepfakeTIMIT_data_results_v2/model_metrics_per_sequence.xlsx', index=False)

# 2. Сохраняем предсказания
np.savez_compressed(
    'balanced_DeepfakeTIMIT_data_results_v2/all_predictions_per_sequence.npz',
    **{name: all_predictions[name]['per_sequence'] for name in all_predictions.keys()}
)

# 3. Сохраняем пороги
np.save('balanced_DeepfakeTIMIT_data_results_v2/optimal_thresholds_per_sequence.npy', all_optimal_thresholds)

# 4. Сохраняем true_labels для последовательностей
np.save('balanced_DeepfakeTIMIT_data_results_v2/true_labels_sequences.npy', true_labels_sequences)

print("Результаты сохранены в директории 'balanced_DeepfakeTIMIT_data_results_v2/'")

# Дополнительно: строим сравнительные графики
print("\nСоздание сравнительных графиков...")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# 1. Сравнение accuracy
ax = axes[0]
models = df_results['model_name']
accuracy = df_results['accuracy']
ax.bar(models, accuracy)
ax.set_title('Accuracy по моделям')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=45)

# 2. Сравнение F1-score
ax = axes[1]
f1_scores = df_results['f1']
ax.bar(models, f1_scores)
ax.set_title('F1-Score по моделям')
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=45)

# 3. Сравнение ROC-AUC
ax = axes[2]
if 'roc_auc' in df_results.columns:
    roc_auc = df_results['roc_auc']
    ax.bar(models, roc_auc)
    ax.set_title('ROC-AUC по моделям')
    ax.set_ylabel('ROC-AUC')
    ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=45)

# 4. Оптимальные пороги
ax = axes[3]
thresholds = df_results['optimal_threshold']
ax.bar(models, thresholds)
ax.set_title('Оптимальные пороги по моделям')
ax.set_ylabel('Порог')
ax.tick_params(axis='x', rotation=45)

# 5. Precision-Recall баланс
ax = axes[4]
precision = df_results['precision']
recall = df_results['recall']
width = 0.35
x = np.arange(len(models))
ax.bar(x - width/2, precision, width, label='Precision')
ax.bar(x + width/2, recall, width, label='Recall')
ax.set_title('Precision и Recall по моделям')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45)
ax.legend()

# 6. Индекс Йудена
ax = axes[5]
youden = df_results['youden_index']
ax.bar(models, youden)
ax.set_title('Индекс Йудена по моделям')
ax.set_ylabel('Индекс Йудена')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('balanced_DeepfakeTIMIT_data_results_v2/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nАнализ завершен!")
print(f"Лучшая модель по accuracy: {df_results.loc[df_results['accuracy'].idxmax(), 'model_name']}")
print(f"Лучшая модель по F1-score: {df_results.loc[df_results['f1'].idxmax(), 'model_name']}")
if 'roc_auc' in df_results.columns:
    print(f"Лучшая модель по ROC-AUC: {df_results.loc[df_results['roc_auc'].idxmax(), 'model_name']}")